# Pokémon TCG AI Battle Challenge: Systematic Exploratory Data Analysis & Deck Strategy Engine
 
Welcome! This notebook provides a professional, detailed, and mathematically rigorous Exploratory Data Analysis (EDA) of the card pool for the [Pokémon TCG AI Battle Challenge Strategy Track](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle-challenge-strategy). 

### Key Technical Implementations
1. **Card ID Deduplication**: Multi-attack Pokémon appear as duplicate rows in the raw CSV. We isolate card-level statistics (HP, Stage, Category) from attack-level statistics to prevent statistical double-counting.
2. **Game-Mechanic Cleansing**: We map null retreat costs to `0.0` (Free Retreat) and parse damage strings into static, scaling (`×` multiplier), and conditional decreasing (`-`) damage types.
3. **Attribute Balance Analysis**: We map HP progression across evolution stages and rules (Normal vs ex vs Mega ex) and plot weakness matrix heatmaps.
4. **Strategy Deck Engine**: We build a complete, rule-compliant TCG deck generator class that automatically traces evolution lines, handles card limits, integrates draw/search Trainer packages, and allocates energy ratios.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Fallback for display() when running as a standard script
try:
    from IPython.display import display
except ImportError:
    display = print

# Configure professional plotting aesthetics
sns.set_theme(style="whitegrid", context="notebook", palette="crest")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

---
## 1. Data Ingestion & Preprocessing
We begin by loading the English card dataset `EN_Card_Data.csv` and cleaning trailing whitespaces.

In [ ]:
# Define paths for Kaggle environment and local execution fallback
data_dir = '/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy'
csv_path = f"{data_dir}/EN_Card_Data.csv"

if not os.path.exists(csv_path):
    print("Running locally or path incorrect. Assuming CSV is in current workspace directory.")
    csv_path = 'EN_Card_Data.csv'

try:
    df = pd.read_csv(csv_path)
    print(f"Dataset successfully loaded. Raw shape: {df.shape}")
except Exception as e:
    print(f"Failed to load dataset: {e}")
    # Demonstration fallback data
    df = pd.DataFrame()

# Clean column names and object columns of whitespaces
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, 'n/a': np.nan})

### Data Cleansing & Parsing Functions
To conduct rigorous analysis, we parse game-mechanic constants:
- **Retreat Costs**: Null values on Pokémon cards represent **Free Retreat (0.0 energy)**.
- **Damage Metrics**: Attacks can deal flat damage (`100`), scaling damage (`30×`), or conditional decreasing damage (`-120`).
- **Attack Energy Costs**: Attacks require specific energy counts (e.g., `{G}●` is 2 energy).

In [ ]:
def parse_damage(dmg_str):
    """
    Parses a damage string into a numeric base value and a damage classification type.
    """
    if pd.isna(dmg_str):
        return np.nan, 'None'
    dmg_str = str(dmg_str).strip()
    if not dmg_str or dmg_str == 'No cost':
        return np.nan, 'None'
    
    # Scaling damage (e.g. 30×)
    if '×' in dmg_str:
        val = re.findall(r'\d+', dmg_str)
        if val:
            return float(val[0]), 'Multiplier'
    # Decreasing/Conditional damage (e.g. -120)
    elif '-' in dmg_str:
        val = re.findall(r'\d+', dmg_str)
        if val:
            return float(val[0]), 'Conditional'
    # Standard static damage
    else:
        val = re.findall(r'\d+', dmg_str)
        if val:
            return float(val[0]), 'Static'
    
    return np.nan, 'None'

def parse_cost_count(cost_str):
    """
    Calculates the total number of energy attachments required for an attack.
    ● represents Colorless energy. Brackets like {G} represent typed energy.
    """
    if pd.isna(cost_str) or cost_str == 'No cost':
        return 0
    cost_str = str(cost_str).strip()
    colorless_count = cost_str.count('●')
    colored_count = cost_str.count('{')
    return colorless_count + colored_count

# Clean and clean specific columns
df['HP'] = pd.to_numeric(df['HP'], errors='coerce')
df['Retreat_Cleaned'] = df['Retreat'].fillna(0.0) # Map NaN retreat to 0.0
df['Type'] = df['Type'].replace('竜', '{Dragon}') # Fix Dragon type encoding

---
## 2. Data Rigor: Card Deduplication
In the raw dataset, a single card appears multiple times if it has multiple attacks or abilities.
 
Analyzing card-level properties (like HP or Stage counts) directly on the raw rows results in severe double-counting bias for cards with multiple attacks. 
 
We explicitly isolate a **deduplicated unique card pool** for card-level statistics.

In [ ]:
df_unique = df.drop_duplicates(subset=['Card ID']).copy()

print(f"Total rows in raw CSV: {len(df)}")
print(f"Total unique cards in set: {len(df_unique)}")
print(f"Double-counted rows avoided: {len(df) - len(df_unique)}")

---
## 3. Card Pool Macro Analysis
Let's inspect the distribution of unique cards by Category (Pokémon, Trainer, Energy) and Trainer sub-types.

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(
    data=df_unique, 
    y='Stage (Pokémon)/Type (Energy and Trainer)', 
    order=df_unique['Stage (Pokémon)/Type (Energy and Trainer)'].value_counts().index,
    hue='Stage (Pokémon)/Type (Energy and Trainer)',
    legend=False
)
plt.title('Distribution of Cards by Category in Card Pool')
plt.xlabel('Number of Unique Cards')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

### Rarity and Rule Types
How common are special card rules (Pokémon ex, Mega Pokémon ex, and ACE SPEC cards)? These rules dictate major deck-building limits and prize-card mechanics.

In [ ]:
rule_counts = df_unique['Rule'].fillna('Normal').value_counts()
print("Card Rule Types Distribution:")
for rule, count in rule_counts.items():
    print(f"- {rule:18s}: {count:4d} cards ({count/len(df_unique)*100:.1f}%)")

---
## 4. Pokémon Attributes & Balance Analysis
We filter our deduplicated set for actual Pokémon cards (excluding Trainers and Energy) to examine HP progression, evolution stages, and retreat costs.

In [ ]:
pok_unique = df_unique[df_unique['Stage (Pokémon)/Type (Energy and Trainer)'].str.contains('Pokémon', na=False) & 
                       ~df_unique['Stage (Pokémon)/Type (Energy and Trainer)'].str.contains('Tool', na=False)].copy()

print(f"Total Unique Pokémon Characters: {len(pok_unique)}")

### HP Distribution & Progression Boxplots
We display the macro HP distribution, and then analyze the linear HP scaling designed by game developers:
- Stage progression: Basic -> Stage 1 -> Stage 2.
- Rule progression: Normal -> Pokémon ex -> Mega Pokémon ex.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: HP Histogram
sns.histplot(data=pok_unique, x='HP', kde=True, bins=25, ax=axes[0], color='#1f77b4')
axes[0].set_title('Overall Pokémon HP Distribution')
axes[0].set_xlabel('Hit Points (HP)')
axes[0].set_ylabel('Count')

# Subplot 2: Stage & Rule Grouped Boxplot
stage_order = ['Basic Pokémon', 'Stage 1 Pokémon', 'Stage 2 Pokémon']
rule_order = ['Normal', 'Pokémon ex', 'Mega Pokémon ex']
pok_stages = pok_unique[pok_unique['Stage (Pokémon)/Type (Energy and Trainer)'].isin(stage_order)].copy()
pok_stages['Rule_Clean'] = pok_stages['Rule'].fillna('Normal')

sns.boxplot(
    data=pok_stages, 
    x='Stage (Pokémon)/Type (Energy and Trainer)', 
    y='HP', 
    hue='Rule_Clean',
    hue_order=rule_order,
    order=stage_order,
    palette='crest',
    ax=axes[1]
)
axes[1].set_title('HP Progression by Stage & Special Rules')
axes[1].set_xlabel('Evolution Stage')
axes[1].set_ylabel('Hit Points (HP)')
axes[1].legend(title='Card Rule')

plt.tight_layout()
plt.show()

### Retreat Cost Profile
We display retreat costs. Note that Free Retreat (0.0) is accurately included due to our preprocessing.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=pok_stages, x='Retreat_Cleaned', hue='Stage (Pokémon)/Type (Energy and Trainer)', palette='crest')
plt.title('Distribution of Pokémon Retreat Costs by Stage')
plt.xlabel('Retreat Cost (Energy Units)')
plt.ylabel('Count')
plt.legend(title='Stage')
plt.tight_layout()
plt.show()

---
## 5. Weakness Heatmap Matrix
In Pokémon TCG, attacking a Pokémon's weakness deals **double damage**, representing a critical tactical match-up.
We create a cross-tabulation matrix of Pokémon Type vs Weakness to find meta-vulnerabilities.

In [ ]:
pok_unique['Weakness_Clean'] = pok_unique['Weakness'].fillna('None')
crosstab = pd.crosstab(pok_unique['Type'], pok_unique['Weakness_Clean'])

plt.figure(figsize=(12, 8))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='crest', cbar=True, linewidths=0.5)
plt.title('Pokémon Type vs Weakness Cross-Tabulation Heatmap')
plt.xlabel('Weakness Type')
plt.ylabel('Pokémon Type')
plt.tight_layout()
plt.show()

> **Competitive Insight**: Dragon-type (`{Dragon}`) Pokémon have **no weakness** in this format, meaning they are immune to type-match double-damage. Fire (`{R}`) and Fighting (`{F}`) represent the most common card-vulnerabilities in the format.

---
## 6. Attack Dynamics & Damage-per-Energy (DPE)
We analyze individual moves and attacks across the dataset. 
We define **Damage-per-Energy (DPE)** as `Base_Damage / Energy_Cost`. A high DPE ratio represents high energy efficiency, enabling quick and cheap knockouts.

In [ ]:
# Extract all attacks (rows containing a move name and an energy cost)
attacks = df[df['Move Name'].notna() & df['Cost'].notna()].copy()
attacks['Energy_Cost'] = attacks['Cost'].apply(parse_cost_count)

# Clean damage values using our parser
parsed_dmg = attacks['Damage'].apply(lambda x: parse_damage(x))
attacks['Base_Damage'] = [x[0] for x in parsed_dmg]
attacks['Damage_Type'] = [x[1] for x in parsed_dmg]

# Calculate DPE for active attacks (excluding 0 energy cost abilities)
attacks_dpe = attacks[(attacks['Energy_Cost'] > 0) & (attacks['Base_Damage'].notna())].copy()
attacks_dpe['DPE'] = attacks_dpe['Base_Damage'] / attacks_dpe['Energy_Cost']

# Plot DPE Histogram
plt.figure(figsize=(10, 4))
sns.histplot(data=attacks_dpe, x='DPE', kde=True, bins=25, color='#4393c3')
plt.title('Attack Efficiency: Damage-per-Energy (DPE) Distribution')
plt.xlabel('DPE (Damage / Energy Cost)')
plt.ylabel('Number of Attacks')
plt.tight_layout()
plt.show()

### Top 10 Most Energy-Efficient Attacks
These attacks offer the highest raw damage return per attached energy card.

In [ ]:
top_dpe = attacks_dpe.sort_values(by='DPE', ascending=False).drop_duplicates(subset=['Card Name']).head(10)
display(top_dpe[['Card Name', 'Move Name', 'Cost', 'Energy_Cost', 'Damage', 'DPE']])

---
## 7. The Strategy Deck-Building Engine
We implement a professional `TCGDeckBuilder` class. It uses recursive logic to construct a legal, fully valid 60-card TCG deck based on a main attacker archetype.
 
### Rules Validated:
1. Total cards must be exactly **60**.
2. Maximum of **4 copies** of any card with the same name (except Basic Energy).
3. Maximum of **1 ACE SPEC** card per deck.
4. Evolution chain completeness (Basic and Stage 1 pre-evolutions must be present for Stage 1/2 cards).

In [ ]:
class TCGDeckBuilder:
    """
    A rule-compliant Pokémon TCG deck generator and validator.
    """
    def __init__(self, df):
        self.df = df.copy()
        
    def trace_evolution(self, pokemon_name):
        """
        Recursively trace evolution stages back to the Basic Pokémon.
        """
        path = [pokemon_name]
        current = pokemon_name
        while True:
            rows = self.df[self.df['Card Name'] == current]
            if rows.empty:
                break
            prev = rows.iloc[0]['Previous stage']
            if pd.isna(prev) or prev == current or prev == 'None':
                break
            prev = str(prev).strip()
            path.append(prev)
            current = prev
        return path

    def build_deck(self, main_attacker, archetype_name):
        """
        Builds a legal 60-card deck themed around a main attacker.
        """
        deck = {}
        evo_line = self.trace_evolution(main_attacker)
        
        # Add evolution line for main attacker (4 Basic, 3 Stage 1, 3 Stage 2/Mega)
        for idx, card in enumerate(reversed(evo_line)):
            if idx == 0:
                deck[card] = 4
            elif idx == 1:
                deck[card] = 3
            else:
                deck[card] = 3
                
        # Archetype specific support cards
        if archetype_name == 'Lightning Blitz':
            deck['Eevee'] = 3
            deck['Jolteon ex'] = 2
            deck['Tapu Koko ex'] = 2
            ace_spec = 'Prime Catcher'
        elif archetype_name == 'Dragon Overlord':
            deck['Raging Bolt ex'] = 2
            deck['Koraidon'] = 2
            ace_spec = 'Sparkling Crystal'
        elif archetype_name == 'Team Rocket Darkness':
            deck["Team Rocket's Mewtwo ex"] = 2
            deck["Team Rocket's Houndour"] = 3
            deck["Team Rocket's Houndoom"] = 2
            ace_spec = 'Legacy Energy'
        else:
            ace_spec = 'Master Ball'
            
        # Check if evolution Stage 2 is present -> Inject Rare Candy
        has_stage_2 = any('Stage 2' in str(row['Stage (Pokémon)/Type (Energy and Trainer)']) 
                          for card in deck 
                          for _, row in self.df[self.df['Card Name'] == card].iterrows())
        if has_stage_2:
            deck['Rare Candy'] = 4
            
        # Draw and Search Engine skeleton
        deck['Ultra Ball'] = 4
        deck['Dusk Ball'] = 2
        
        # Search low-HP basics with Buddy-Buddy Poffin (Basics with <= 70 HP)
        has_low_hp_basic = False
        for card in deck:
            rows = self.df[self.df['Card Name'] == card]
            if not rows.empty:
                stage = str(rows.iloc[0]['Stage (Pokémon)/Type (Energy and Trainer)'])
                hp = pd.to_numeric(rows.iloc[0]['HP'], errors='coerce')
                if 'Basic' in stage and hp <= 70:
                    has_low_hp_basic = True
                    break
        if has_low_hp_basic:
            deck['Buddy-Buddy Poffin'] = 3
            
        deck['Boss’s Orders'] = 2  # Gust effect
        deck['Carmine'] = 4        # Draw supporter
        deck['Judge'] = 2          # Disruption/Draw
        deck['Night Stretcher'] = 2 # Recovery
        deck[ace_spec] = 1         # 1 ACE SPEC limit
        
        # Calculate deck sum and fill remaining slots with Energy
        current_sum = sum(deck.values())
        needed_energy = 60 - current_sum
        
        if needed_energy > 0:
            if archetype_name == 'Lightning Blitz':
                deck['Basic {L} Energy'] = needed_energy
            elif archetype_name == 'Dragon Overlord':
                w_energy = needed_energy // 3
                f_energy = needed_energy // 3
                l_energy = needed_energy - w_energy - f_energy
                deck['Basic {W} Energy'] = w_energy
                deck['Basic {F} Energy'] = f_energy
                deck['Basic {L} Energy'] = l_energy
            elif archetype_name == 'Team Rocket Darkness':
                deck["Team Rocket's Energy"] = min(4, needed_energy)
                remaining = needed_energy - deck["Team Rocket's Energy"]
                if remaining > 0:
                    d_energy = remaining // 2
                    p_energy = remaining - d_energy
                    deck['Basic {D} Energy'] = d_energy
                    deck['Basic {P} Energy'] = p_energy
            else:
                deck['Basic {C} Energy'] = needed_energy
                
        self.validate_deck(deck)
        return deck
        
    def validate_deck(self, deck):
        """
        Validates all deck building criteria against standard Pokémon TCG tournament rules.
        """
        total_cards = sum(deck.values())
        assert total_cards == 60, f"Deck must have exactly 60 cards, found {total_cards}."
        
        for card, qty in deck.items():
            if 'Basic' in card and 'Energy' in card:
                continue
            assert qty <= 4, f"Card '{card}' has {qty} copies, exceeding the limit of 4."
            
        ace_specs = []
        for card in deck:
            rows = self.df[self.df['Card Name'] == card]
            if not rows.empty:
                rule = str(rows.iloc[0]['Rule'])
                if 'ACE SPEC' in rule:
                    ace_specs.append(card)
        assert len(ace_specs) <= 1, f"Deck contains multiple ACE SPEC cards: {ace_specs}."
        
        for card in deck:
            rows = self.df[self.df['Card Name'] == card]
            if not rows.empty:
                prev = rows.iloc[0]['Previous stage']
                if not pd.isna(prev) and prev != 'None' and prev != card:
                    stage = str(rows.iloc[0]['Stage (Pokémon)/Type (Energy and Trainer)'])
                    if 'Stage 1' in stage:
                        assert prev in deck, f"Evolution validation failed: '{card}' requires '{prev}' in the deck."
                    elif 'Stage 2' in stage:
                        basic_stage = self.trace_evolution(card)[-1]
                        assert basic_stage in deck, f"Evolution validation failed: '{card}' requires Basic '{basic_stage}' in the deck."

### Generating Decks for Top Archetypes
We run our deck-building engine for three separate archetypes, printing details of the deck lists and confirmation of their validation status.

In [ ]:
builder = TCGDeckBuilder(df)

archetypes = [
    ("Pikachu ex", "Lightning Blitz"),
    ("Mega Dragonite ex", "Dragon Overlord"),
    ("Team Rocket's Houndoom", "Team Rocket Darkness")
]

for mon, name in archetypes:
    print(f"\n==================================================")
    print(f"🧬 ARCHETYPE: {name.upper()}")
    print(f"==================================================")
    deck_list = builder.build_deck(mon, name)
    
    # Sort card names by type for a professional deck layout
    sorted_deck = sorted(deck_list.items(), key=lambda x: x[0])
    for card, qty in sorted_deck:
        print(f"{qty}x {card}")
    print(f"\n✓ Validation Status: Deck matches all tournament regulations.")

---
## Conclusion & Next Steps
 
This EDA sets a rigorous, deduplicated foundation for analyzing the Pokémon TCG card pool. 
 
- Decks built around **Dragon-type Pokémon** (like `Mega Dragonite ex`) benefit from having zero weaknesses, but require complex dual-type energy systems.
- Decks built around **Lightning-type Pokémon** (like `Pikachu ex`) offer high DPE ratios and rapid bench setups.
- Decks built around **Team Rocket Pokémon** benefit from dedicated support like `Team Rocket's Energy` supplying multiple energy types.
 
You can use the `TCGDeckBuilder` class in this notebook to model and simulate the success rate of various deck skeletons. Best of luck in the challenge!